# 07 - Logit lens v2 (hook-based, validated)

Regenerates the logit-lens curves taking residual states via **forward hooks
during the live pass**, rather than reconstructing from float32-stored hidden
states.

## Why

The v1 path (`capture_activations` -> store float32 -> re-apply `final_norm` and
`lm_head`) disagrees with the model's own final logits by up to ~1.5 units. The
hook-based path matches to five decimal places. The validation cell below is the
proof: at the last layer, the lens value must equal the model's own output.

Curve *shape* and the text/image/control ordering are unchanged between versions;
only the absolute values move. Use these curves for the paper.

Note the indexing convention: layers are 0-indexed over 28 decoder blocks, so
emergence is at **layer 27 of 28**. The v1 code included the embedding as layer 0
over 29 positions, which is why it reported "layer 28". Same layer, different
label - state the convention in any figure caption.

Produces `results/logit_lens_v2/`.

> Reconstructed from session transcripts; outputs not embedded.

## Setup

Clone the repo, install deps, load Qwen2.5-VL-7B in bf16 across 2x T4.

**Check Accelerator = GPU T4 x2 before running.** A Kaggle batch job inherits
`None` silently and runs at ~1200 s/item on CPU. The assert below catches it.

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GH_TOKEN")
import os, sys
if not os.path.isdir("/kaggle/working/algoverse"):
    !git clone https://{token}@github.com/bryantran21/algoverse.git /kaggle/working/algoverse
sys.path.insert(0, "/kaggle/working/algoverse"); os.chdir("/kaggle/working/algoverse")
!git config user.email "bryantran21@gmail.com"
!git config user.name  "bryantran21"

import subprocess
subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.49.0',
    'accelerate>=0.34.0','datasets','qwen-vl-utils','typst','Pillow',
    'scikit-learn','matplotlib','tqdm'], check=True)

import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.device_count(), "devices")
assert torch.cuda.is_available(), "NO GPU - set Accelerator to T4 x2 before running"

import config
config.DEVICE_MAP = "auto"        # 2-GPU full precision
config.LOAD_IN_4BIT = False       # 4-bit perturbs activations - never use for interp
from src.inference import load_vl_model
model, processor = load_vl_model()
print("READY", flush=True)

## Hook-based lens + validation

The last printed `diff` must be ~0. If it is not, do not trust the curves.

In [ ]:
import config, torch, gc, numpy as np, pickle, os
gc.collect(); torch.cuda.empty_cache()
os.makedirs("results/logit_lens_v2", exist_ok=True)

from src.inference import _messages_for, _flatten_images
YES, NO = 9454, 2753
layers = model.language_model.layers if hasattr(model, "language_model") \
         else model.model.language_model.layers
lm_head    = model.get_output_embeddings()
final_norm = dict(model.named_modules())["model.language_model.norm"]
norm_dev   = next(final_norm.parameters()).device
head_dev   = next(lm_head.parameters()).device

sets = pickle.load(open("results/sets/sets_n2000.pkl","rb"))
fail_all, ctrl_all = sets["fail_all"], sets["ctrl_all"]
config.RENDER["font_size_pt"] = 5.0

@torch.no_grad()
def lens_curve(item, mode):
    """P(gold token) at every layer, read live from the forward pass."""
    msgs, images = _messages_for(item, mode)
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    imgs = _flatten_images([images])
    inputs = processor(text=[text], images=imgs if imgs else None,
                       return_tensors="pt").to(model.device)

    store, hooks = {}, []
    for i, layer in enumerate(layers):
        def mk(i):
            def hook(mod, inp, out):
                h = out[0] if isinstance(out, tuple) else out
                store[i] = h[0, -1, :].detach()
            return hook
        hooks.append(layer.register_forward_hook(mk(i)))
    out = model(**inputs, use_cache=False)
    for h in hooks: h.remove()

    gold_id = YES if item["gold"] == "yes" else NO
    probs = []
    for i in range(len(layers)):
        v  = store[i].to(norm_dev)
        lg = lm_head(final_norm(v).to(head_dev)).float()
        probs.append(torch.softmax(lg, dim=-1)[gold_id].item())
    final_p = torch.softmax(out.logits[0, -1].float(), dim=-1)[gold_id].item()
    return np.array(probs), final_p

for it in ctrl_all[:3]:
    c_, fp = lens_curve(it, "image")
    print(f"item {it['item_id']}: lens L27 {c_[-1]:.4f}  model final {fp:.4f}  "
          f"diff {abs(c_[-1]-fp):.5f}", flush=True)

## Full run

Three curves: text on the failure set, image on the failure set, and image on the
control set. The control curve is what makes the failure curves interpretable -
without it, a flat image-mode curve says nothing, since the failure set is
*selected* on image-wrong.

In [ ]:
import time
N_ITEMS = 62
t0 = time.time()
txt_c, img_c, ctrl_c = [], [], []

for k, it in enumerate(fail_all[:N_ITEMS]):
    txt_c.append(lens_curve(it, "text")[0])
    img_c.append(lens_curve(it, "image")[0])
    torch.cuda.empty_cache()
    if (k+1) % 20 == 0: print(f"fail {k+1}/{N_ITEMS} ({(time.time()-t0)/60:.1f}m)", flush=True)

for k, it in enumerate(ctrl_all[:N_ITEMS]):
    ctrl_c.append(lens_curve(it, "image")[0])
    torch.cuda.empty_cache()
    if (k+1) % 20 == 0: print(f"ctrl {k+1}/{N_ITEMS} ({(time.time()-t0)/60:.1f}m)", flush=True)

txt_c, img_c, ctrl_c = map(np.array, (txt_c, img_c, ctrl_c))
np.savez("results/logit_lens_v2/curves.npz", txt=txt_c, img=img_c, ctrl_img=ctrl_c)

def boot(C, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    b = C[rng.integers(0, len(C), size=(n, len(C)))].mean(axis=1)
    return C.mean(axis=0), np.percentile(b, 2.5, axis=0), np.percentile(b, 97.5, axis=0)

for name, C in (("text/fail", txt_c), ("image/fail", img_c), ("image/ctrl", ctrl_c)):
    m = C.mean(axis=0)
    h = np.where(m > 0.5)[0]
    print(f"{name:12s} emerges layer {h[0] if len(h) else None}  peak {m.max():.3f}", flush=True)

In [ ]:
import matplotlib.pyplot as plt
L = np.arange(txt_c.shape[1])
plt.figure(figsize=(8,4.5))
for C, col, mk, lbl in [(txt_c,"green","o-",f"text, failure set (n={len(txt_c)})"),
                        (img_c,"crimson","s-",f"image, failure set (n={len(img_c)})"),
                        (ctrl_c,"steelblue","^-",f"image, control set (n={len(ctrl_c)})")]:
    m, lo, hi = boot(C)
    plt.plot(L, m, mk, color=col, ms=4, label=lbl)
    plt.fill_between(L, lo, hi, color=col, alpha=0.15)
plt.axhline(0.5, color="gray", ls=":")
plt.xlabel("layer (0-indexed over 28 decoder blocks)")
plt.ylabel("P(correct answer) via logit lens")
plt.title("Where the answer emerges - Qwen2.5-VL-7B, BoolQ, 5pt")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("results/logit_lens_v2/logit_lens.png", dpi=150); plt.show()

In [ ]:
!git add -A && git commit -m "07: hook-based logit lens, validated against model logits" && git push origin master